In [ ]:
from google.colab import drive
import os

# Mount Drive
drive.mount('/content/drive')

# 2. Unzip the file
!unzip -q "/content/drive/MyDrive/cattle/archive (3).zip" -d /content/raw_data

print("Unzipping complete!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Unzipping complete!


In [ ]:
import os
import shutil
!pip install split-folders
raw_path = "/content/raw_data"
subfolders = [f for f in os.listdir(raw_path) if os.path.isdir(os.path.join(raw_path, f))]

if len(subfolders) == 1:
    hidden_folder = os.path.join(raw_path, subfolders[0])
    print(f"Found hidden folder: {hidden_folder}. Fixing path...")
    input_path = hidden_folder
else:
    input_path = raw_path

if os.path.exists('CattleDataset'):
    shutil.rmtree('CattleDataset')


import splitfolders
splitfolders.ratio(input_path, output="CattleDataset", seed=1337, ratio=(.8, .2))


train_dir = '/content/CattleDataset/train'
if os.path.exists(train_dir):
    actual_breeds = os.listdir(train_dir)
    print(f"✅ SUCCESS! Total breeds found: {len(actual_breeds)}")
    if len(actual_breeds) > 1:
        print(f"Example breeds detected: {actual_breeds[:5]}")
else:
    print("❌ Still not seeing folders. Please run '!ls -R /content/raw_data' and tell me what you see.")

Found hidden folder: /content/raw_data/cattle. Fixing path...


Copying files: 8539 files [00:22, 386.68 files/s]

✅ SUCCESS! Total breeds found: 50
Example breeds detected: ['Bargur', 'Punganur', 'Amritmahal', 'badri', 'gangatari']


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import os


device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}


data_dir = '/content/CattleDataset'
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x])
                  for x in ['train', 'val']}
dataloaders = {x: DataLoader(image_datasets[x], batch_size=32, shuffle=True, num_workers=2)
              for x in ['train', 'val']}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes

model = models.mobilenet_v3_small(weights='DEFAULT')

# Updating the final layer for 50 breeds
num_ftrs = model.classifier[3].in_features
model.classifier[3] = nn.Linear(num_ftrs, len(class_names))
model = model.to(device)

#  Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training Loop
num_epochs = 10
print(f"Starting Training for {len(class_names)} breeds...")

for epoch in range(num_epochs):
    print(f'Epoch {epoch+1}/{num_epochs}')
    print('-' * 10)

    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()
        else:
            model.eval()

        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in dataloaders[phase]:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        epoch_loss = running_loss / dataset_sizes[phase]
        epoch_acc = running_corrects.double() / dataset_sizes[phase]

        print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

# Saving the model
torch.save(model.state_dict(), 'cattle_breed_model.pth')
print("✅ SUCCESS! Your brain is trained. Download 'cattle_breed_model.pth' from the sidebar.")

Using device: cuda:0
Starting Training for 50 breeds...
Epoch 1/10
----------


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


train Loss: 2.4836 Acc: 0.3071
val Loss: 2.0778 Acc: 0.3917
Epoch 2/10
----------


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


train Loss: 1.9085 Acc: 0.4322
val Loss: 1.6715 Acc: 0.5000
Epoch 3/10
----------


...

train Loss: 1.3605 Acc: 0.5781
val Loss: 1.5392 Acc: 0.5585
Epoch 9/10
----------


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


train Loss: 1.2685 Acc: 0.6113
val Loss: 1.5416 Acc: 0.5637
Epoch 10/10
----------


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


train Loss: 1.2388 Acc: 0.6120
val Loss: 1.4412 Acc: 0.5811
✅ SUCCESS! Your brain is trained. Download 'cattle_breed_model.pth' from the sidebar.


In [ ]:

class_names = sorted(os.listdir('/content/CattleDataset/train'))

with open('classes.txt', 'w') as f:
    for breed in class_names:
        f.write(f"{breed}\n")

print("✅ classes.txt created! Refresh the sidebar to see it.")

✅ classes.txt created! Refresh the sidebar to see it.
